# Lezione 09 — Test di ipotesi III: categorie e confronti multipli

**Corso:** Statistica Applicata con Python · **Blocco 3:** Inferenza
**Tempo stimato:** 105 minuti · **Capitolo del manuale:** `manual/09_tests_hipotesis_iii_IT.md`

## Cosa vedrai in questa lezione

- **Dati categoriali:** tabelle di contingenza e **chi-quadro di indipendenza** (la tabella 2×2 a mano, frequenze attese, gradi di libertà), **V di Cramér**, residui e bontà di adattamento.
- **Il legame con la lezione 7:** in una tabella 2×2, **z² = χ²** (test per due proporzioni e chi-quadro sono lo stesso test).
- **Quando il chi-quadro non funziona:** la condizione sulle frequenze attese e il **test esatto di Fisher**.
- **Confronti multipli:** perché 20 test danno quasi sempre una "scoperta" falsa, **Bonferroni** (FWER) e **Benjamini-Hochberg** (FDR).
- **Come applicare le correzioni:** ai post-hoc della lezione 8 (Tukey) e a molti chi-quadro su categorie o sottogruppi diversi (p-hacking).
- **Una tabella per decidere** quale test usare (lezioni 7-9).

## Prerequisiti

- Lezione 7 (p-value, errori di tipo I e II, test per due proporzioni) e lezione 8 (ANOVA e post-hoc di Tukey).
- Librerie: `numpy`, `pandas`, `matplotlib`, `scipy`, `statsmodels` (`pip install numpy pandas matplotlib scipy statsmodels`; sono già installate in Google Colab).

> **Come usare questo notebook:** esegui le celle in ordine. Prima di ogni esperimento scrivi cosa ti aspetti che accada. Tutti i dati (visite per canale e conversioni, tempi di preparazione) sono sintetici: un *esempio illustrativo*, non dati reali.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo il **seme**: con lo stesso seme, tutti otteniamo gli stessi numeri.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest

rng = np.random.default_rng(42)          # seme fisso: tutti otteniamo gli stessi risultati
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True
pd.set_option("display.width", 250)
pd.set_option("display.max_columns", None)
print("Librerie caricate correttamente.")

---
## 1. I dati: canale di acquisizione × conversione

Un negozio online acquisisce visite da tre canali: email, social e motore di ricerca. Per ogni visita conosciamo il canale e se ha **acquistato** o no. Le due variabili sono **categoriali**, quindi non ci sono medie da confrontare: bisogna **contare** (*esempio illustrativo*, dati sintetici, 200 visite per canale).

Il riassunto naturale di due variabili categoriali è la **tabella di contingenza**: una cella per ogni combinazione di categorie, con il **conteggio** dei casi.

In [ ]:
canales = ["Email", "Social", "Motore di ricerca"]
compras = [48, 70, 42]                                   # esempio illustrativo: acquisti su 200 visite per canale
visitas = pd.DataFrame({
    "canal": np.repeat(canales, 200),
    "compra": np.concatenate([np.repeat(["sì", "no"], [k, 200 - k]) for k in compras])})   # 600 righe: una per visita
print(visitas.head(3).to_string(), "\n")

tabla = pd.crosstab(visitas["canal"], visitas["compra"]).loc[canales, ["sì", "no"]]
display(tabla)
print("Tasso di acquisto per canale (%):")
print((tabla["sì"] / tabla.sum(axis=1) * 100).round(1).to_string())

---
## 2. Chi-quadro di indipendenza: la tabella 2×2 a mano

Cominciamo con due canali e una tabella piccola, per poter calcolare tutto a mano. H0: canale e acquisto sono **indipendenti** (il tasso di acquisto è lo stesso). Sotto H0, la **frequenza attesa** di ogni cella è (totale di riga × totale di colonna) / totale generale; la statistica è **Σ (osservata − attesa)² / attesa**, con (righe − 1)·(colonne − 1) gradi di libertà.

In [ ]:
observada = np.array([[30, 70],
                      [45, 55]])          # righe: canale 1, canale 2; colonne: acquista, non acquista (100 visite per canale)
tot_fila = observada.sum(axis=1, keepdims=True)
tot_col = observada.sum(axis=0, keepdims=True)
esperada = tot_fila * tot_col / observada.sum()
chi2_mano = ((observada - esperada) ** 2 / esperada).sum()
gl_chi = (observada.shape[0] - 1) * (observada.shape[1] - 1)
p_chi = stats.chi2.sf(chi2_mano, gl_chi)
print("Attese sotto H0:\n", esperada)
print(f"a mano: χ² = {chi2_mano:.3f}   gl = {gl_chi}   p = {p_chi:.4f}")

sin_corr = stats.chi2_contingency(observada, correction=False)
con_corr = stats.chi2_contingency(observada)                      # per default nel 2×2: correzione di Yates
print(f"scipy senza correzione: χ² = {sin_corr.statistic:.3f}   p = {sin_corr.pvalue:.4f}")
print(f"scipy con correzione di Yates: χ² = {con_corr.statistic:.3f}   p = {con_corr.pvalue:.4f}")
assert np.isclose(chi2_mano, sin_corr.statistic) and np.isclose(p_chi, sin_corr.pvalue)

v_cramer = np.sqrt(chi2_mano / (observada.sum() * (min(observada.shape) - 1)))
print(f"V di Cramér = {v_cramer:.3f}")

**Cosa osservi?**

- Le attese sono 37,5 e 62,5 in ogni riga (il tasso globale di acquisto è 75/200 = 37,5 %). Le osservate se ne discostano di ±7,5.
- χ² = 4,80 con 1 gdl → p ≈ 0,028 (senza correzione). Con la correzione di Yates viene 4,18 e p ≈ 0,041: per tabelle 2×2 piccole è la versione più prudente. Dichiara sempre quale usi.
- V di Cramér ≈ 0,15: associazione presente ma debole. Il p-value non dice quanto è forte.

*(La tua risposta qui.)*

---
## 3. Il legame con la lezione 7: z² = χ²

Nella lezione 7 hai confrontato due proporzioni con `proportions_ztest` (varianza combinata). Una tabella 2×2 **è** un confronto tra due proporzioni: "canale 1 contro canale 2" nella colonna "acquisto". Verifichiamo che i due test sono lo stesso test, anche con l'esempio della lezione 7 (244 su 4 000 contro 200 su 4 000).

In [ ]:
z, p_z = proportions_ztest([30, 45], [100, 100])
print(f"z = {z:.4f}   p = {p_z:.4f}   z² = {z ** 2:.4f}   χ² ({'senza correzione'}) = {sin_corr.statistic:.4f}")
assert np.isclose(z ** 2, sin_corr.statistic) and np.isclose(p_z, sin_corr.pvalue)

# L'esempio della lezione 7 (esempio illustrativo): 244 su 4 000 contro 200 su 4 000
z7, p7 = proportions_ztest([244, 200], [4000, 4000])
t7 = np.array([[244, 4000 - 244], [200, 4000 - 200]])
c7 = stats.chi2_contingency(t7, correction=False)
print(f"Lezione 7: z = {z7:.3f}   p = {p7:.4f}   z² = {z7 ** 2:.3f}   χ² = {c7.statistic:.3f}   p(χ²) = {c7.pvalue:.4f}")
assert np.isclose(z7 ** 2, c7.statistic) and np.isclose(p7, c7.pvalue)

# E la correzione di Yates? Non coincide con z² (è una correzione in più del chi-quadro)
print(f"Con Yates: χ² = {con_corr.statistic:.3f}  (≠ z² = {z ** 2:.3f})")

### Esperimento guidato: z² = χ² sempre?

Generiamo 200 tabelle 2×2 casuali (dimensioni dei gruppi e tassi a caso) e confrontiamo, in ciascuna, z² con il χ² senza correzione.

In [ ]:
rng_z = np.random.default_rng(9)
dif_max = 0.0
for _ in range(200):
    n1, n2 = rng_z.integers(30, 500, size=2)
    a, b = rng_z.binomial(n1, rng_z.uniform(0.1, 0.5)), rng_z.binomial(n2, rng_z.uniform(0.1, 0.5))
    if a + b in (0, n1 + n2):
        continue
    zz, _ = proportions_ztest([a, b], [n1, n2])
    chi = stats.chi2_contingency(np.array([[a, n1 - a], [b, n2 - b]]), correction=False).statistic
    dif_max = max(dif_max, abs(zz ** 2 - chi))
print(f"Differenza massima |z² − χ²| in 200 tabelle = {dif_max:.2e}")
assert dif_max < 1e-8

**Cosa osservi?**

- z² e χ² coincidono fino all'errore di arrotondamento: **sono lo stesso test** (con la stessa varianza combinata) e danno lo stesso p-value a due code.
- Allora a cosa servono entrambi? Lo **z** conserva il **segno** (quale gruppo converte di più), permette test a **una coda** e dà la differenza tra i tassi con il suo IC. Il **χ²** si generalizza a **più di due categorie** (tabelle 3×2, 4×5...), dove non c'è una sola differenza.

*(La tua risposta qui.)*

---
## 4. Tabelle più grandi, residui e bontà di adattamento

Con **tre** canali la tabella è 3×2 e i gradi di libertà sono (3 − 1)·(2 − 1) = 2. Per vedere **dove** si concentra l'associazione guardiamo i **residui standardizzati** (osservata − attesa) / √attesa: i valori con modulo maggiore di 2 spiccano.

Il **chi-quadro di bontà di adattamento** confronta **una** variabile categoriale con una distribuzione attesa: per esempio, gli incidenti si ripartiscono in modo uguale tra tre turni?

In [ ]:
obs3 = tabla.to_numpy()
r3 = stats.chi2_contingency(obs3)
print(f"χ² = {r3.statistic:.3f}   gl = {r3.dof}   p = {r3.pvalue:.4f}   V di Cramér = {np.sqrt(r3.statistic / (obs3.sum() * (min(obs3.shape) - 1))):.3f}")
print("Attese (tutte ≥ 5):", r3.expected_freq[0].round(2), "minima =", r3.expected_freq.min().round(2))
resid = (obs3 - r3.expected_freq) / np.sqrt(r3.expected_freq)
display(pd.DataFrame(resid.round(2), index=canales, columns=["acquista", "non acquista"]))

tasas = obs3[:, 0] / obs3.sum(axis=1)
plt.bar(canales, tasas * 100, color=["tab:blue", "tab:orange", "tab:green"])
plt.axhline(obs3[:, 0].sum() / obs3.sum() * 100, color="black", linestyle="--", label="tasso globale")
plt.ylabel("% di visite che acquistano"); plt.title("Conversione per canale"); plt.legend()
plt.show()

# Bontà di adattamento: incidenti per turno (atteso: ripartizione uguale)
gof = stats.chisquare([30, 20, 10])
print(f"Bontà di adattamento: χ² = {gof.statistic:.2f}   p = {gof.pvalue:.4f}   (gl = 2, attese 20-20-20)")

**Cosa osservi?**

- p ≈ 0,004: i tassi di conversione differiscono tra i canali. V ≈ 0,14: associazione debole ma reale.
- Il residuo maggiore è in "Social – acquisto" (2,28: più acquisti di quelli attesi); il resto resta sotto 2 in valore assoluto.
- Come un'ANOVA, il chi-quadro globale **non dice quali canali differiscono**: i residui sono un indizio descrittivo, non un test post-hoc. Per confrontare coppie servirà correggere (sezione 8).
- La bontà di adattamento rifiuta la ripartizione uguale tra i turni (χ² = 10, p ≈ 0,007), ma non dice in quale turno si concentrano gli incidenti.

*(La tua risposta qui.)*

---
## 5. La condizione del chi-quadro: frequenze attese e test esatto di Fisher

Il chi-quadro è un'**approssimazione** che funziona bene se le frequenze **attese** (non quelle osservate) non sono minuscole. **Regola pratica:** tutte ≥ 5 (alcuni testi accettano che solo una piccola parte sia tra 1 e 5, ma nessuna sotto 1). Se non vale, in una 2×2 si usa il **test esatto di Fisher**, che calcola il p-value esatto senza approssimare. Inoltre il chi-quadro richiede **conteggi**, non percentuali: con le percentuali la dimensione del campione sparisce e il risultato non ha senso.

In [ ]:
pequena = np.array([[3, 9],
                    [7, 2]])              # esempio illustrativo: 21 casi
rc = stats.chi2_contingency(pequena)
print("Attese:\n", rc.expected_freq.round(2), "\nminima attesa =", rc.expected_freq.min().round(2), "(< 5)")
print(f"χ² (con Yates): p = {rc.pvalue:.4f}")
print(f"Fisher esatto:      p = {stats.fisher_exact(pequena).pvalue:.4f}   odds ratio = {stats.fisher_exact(pequena).statistic:.3f}")

### Esperimento guidato: il tasso esatto di falsi positivi con tabelle piccole

Due gruppi di dimensione 5 e 25 con lo **stesso** tasso di successo vero (5 %; H0 vera). Calcoliamo, senza simulazione, la **probabilità esatta** che ciascun test dia p < 0,05: sommiamo la probabilità binomiale di ogni tabella possibile in cui il test rifiuta. Un test ben calibrato dovrebbe dare 0,05 (o meno).

In [ ]:
n1_t, n2_t, p_ver = 5, 25, 0.05
pm1 = stats.binom.pmf(np.arange(n1_t + 1), n1_t, p_ver)
pm2 = stats.binom.pmf(np.arange(n2_t + 1), n2_t, p_ver)
fpr = {"χ² senza correzione": 0.0, "χ² con Yates": 0.0, "Fisher": 0.0}
for a in range(n1_t + 1):
    for b in range(n2_t + 1):
        if a + b == 0 or a + b == n1_t + n2_t:
            continue                                   # tabella senza variazione: non si può testare
        tab = np.array([[a, n1_t - a], [b, n2_t - b]])
        w = pm1[a] * pm2[b]
        if stats.chi2_contingency(tab, correction=False).pvalue < 0.05: fpr["χ² senza correzione"] += w
        if stats.chi2_contingency(tab, correction=True).pvalue < 0.05: fpr["χ² con Yates"] += w
        if stats.fisher_exact(tab).pvalue < 0.05: fpr["Fisher"] += w
for nombre, valor in fpr.items():
    print(f"{nombre:<28} tasso esatto di falsi positivi = {valor:.4f}")

**Cosa osservi?**

- Con attese minuscole, il chi-quadro senza correzione rifiuta una H0 vera circa il 7 % delle volte: **più** del 5 % nominale. L'approssimazione non è affidabile.
- Yates e Fisher sono conservativi (≈ 0,7 %): proteggono dai falsi positivi a costo di potenza.
- Morale: controlla le attese prima di fidarti del p-value del chi-quadro; con 2×2 piccole, Fisher.

*(La tua risposta qui.)*

---
## 6. Confronti multipli: perché 20 test danno quasi sempre una "scoperta"

Con **un** test e H0 vera, la probabilità di un falso positivo è α = 0,05. Con **m** test indipendenti e H0 vera in tutti, la probabilità di **almeno un** falso positivo è **1 − (1 − α)^m**. È il **tasso di errore per famiglia** (FWER, *family-wise error rate*).

In [ ]:
alfa = 0.05
tabla_m = pd.DataFrame({"m": [1, 3, 5, 10, 15, 20, 50]})
tabla_m["P(almeno un falso positivo)"] = 1 - (1 - alfa) ** tabla_m["m"]
display(tabla_m.set_index("m").round(3))
print(f"m = 20: 1 − 0,95^20 = {1 - 0.95 ** 20:.4f}")

### Esperimento guidato: 20 confronti tra gruppi con la stessa distribuzione

Ripetiamo 5 000 volte uno "studio" con **20 confronti**; in ciascuno i due gruppi (30 dati) provengono dalla **stessa** distribuzione normale, quindi **tutte le H0 sono vere**. Contiamo quanti test risultano "significativi" a α = 0,05, senza correzione, con Bonferroni e con Benjamini-Hochberg.

In [ ]:
rng_mc = np.random.default_rng(42)
REPS, M = 5000, 20
datos_mc = rng_mc.normal(size=(REPS, M, 2, 30))
p_mc = stats.ttest_ind(datos_mc[:, :, 0, :], datos_mc[:, :, 1, :], axis=2).pvalue      # forma (REPS, M)

sin_corregir = (p_mc < 0.05)
bonf = (p_mc < 0.05 / M)
bh = np.array([multipletests(fila, alpha=0.05, method="fdr_bh")[0] for fila in p_mc])

print(f"Teoria: 1 − 0,95^20 = {1 - 0.95 ** M:.4f}")
print(f"Senza correzione: studi con almeno una 'scoperta' = {sin_corregir.any(axis=1).mean():.4f}")
print(f"Senza correzione: falsi positivi per studio (media) = {sin_corregir.sum(axis=1).mean():.3f}")
print(f"Bonferroni (α/m = {0.05 / M:.4f}): studi con qualche falso positivo = {bonf.any(axis=1).mean():.4f}")
print(f"Benjamini-Hochberg: studi con qualche falso positivo = {bh.any(axis=1).mean():.4f}")

se = np.sqrt(0.6415 * (1 - 0.6415) / REPS)
assert abs(sin_corregir.any(axis=1).mean() - (1 - 0.95 ** M)) < 4 * se, "la simulazione non coincide con la teoria"

plt.hist(sin_corregir.sum(axis=1), bins=np.arange(-0.5, 8.5, 1), edgecolor="white")
plt.title("Falsi positivi per studio (20 test, tutte le H0 vere)")
plt.xlabel("numero di test 'significativi'"); plt.ylabel("numero di studi")
plt.show()

**Cosa osservi?**

- Senza correzione, ≈ 64 % degli studi trova almeno una "scoperta" anche se **non c'è nulla**: coincide con 1 − 0,95²⁰ = 0,642. In media compare ≈ 1 falso positivo per studio (20 · 0,05).
- Con Bonferroni e con Benjamini-Hochberg, quando tutte le H0 sono vere la probabilità di qualche falso positivo scende a ≈ 5 %.
- Se il team guarda 20 sottogruppi e comunica solo quello che "esce", sta facendo proprio questo (*p-hacking*).

*(La tua risposta qui.)*

---
## 7. Bonferroni contro FDR di Benjamini-Hochberg

- **Bonferroni:** rifiuta solo se p ≤ α/m. Controlla la **FWER**: la probabilità di **qualsiasi** falso positivo. Molto conservativo con molti test.
- **Benjamini-Hochberg (BH):** ordina i p-value e confronta l'i-esimo con (i/m)·α; rifiuta tutti fino al maggiore che soddisfa la condizione. Controlla la **FDR** (*false discovery rate*): la **proporzione attesa di falsi tra le scoperte**. Più potente; ammette qualche falso positivo.
- **Holm** (`method="holm"`) è un Bonferroni a scalini: controlla anch'esso la FWER e **non** è mai meno potente.

Prima, con 8 p-value di esempio (ordinati); poi, un esperimento con 20 test di cui 5 con effetto reale.

In [ ]:
pv = np.array([0.001, 0.008, 0.012, 0.021, 0.040, 0.049, 0.20, 0.50])
m = len(pv)
rech_b, p_aj_b, _, umbral_b = multipletests(pv, alpha=0.05, method="bonferroni")
rech_h, p_aj_h, _, _ = multipletests(pv, alpha=0.05, method="fdr_bh")
umbral_bh = np.arange(1, m + 1) / m * 0.05
display(pd.DataFrame({"p": pv, "soglia BH (i/m·α)": umbral_bh,
                      "p corretto Bonferroni": p_aj_b, "rifiuta Bonferroni": rech_b,
                      "p corretto BH": p_aj_h, "rifiuta BH": rech_h}).round(4))
print(f"Senza correzione rifiutano {(pv < 0.05).sum()};  Bonferroni (α/m = {0.05 / m:.5f}) {rech_b.sum()};  BH {rech_h.sum()}")

In [ ]:
rng_fdr = np.random.default_rng(5)
REPS_F, M_F, N_REALES = 2000, 20, 5
acum = {"Senza correzione": [], "Bonferroni": [], "Benjamini-Hochberg": []}
reales = np.arange(M_F) < N_REALES
for _ in range(REPS_F):
    a = rng_fdr.normal(size=(M_F, 30))
    b = rng_fdr.normal(size=(M_F, 30))
    b[:N_REALES] += 0.9                                 # 5 confronti con effetto reale di 0,9 σ
    pvs = stats.ttest_ind(a, b, axis=1).pvalue
    decisiones = [pvs < 0.05, multipletests(pvs, method="bonferroni")[0], multipletests(pvs, method="fdr_bh")[0]]
    for clave, rech in zip(acum, decisiones):
        acum[clave].append((rech.sum(), (rech & ~reales).sum(), (rech & reales).sum()))

filas = []
for clave, v in acum.items():
    v = np.array(v)
    filas.append({"metodo": clave,
                  "scoperte (media)": v[:, 0].mean(),
                  "falsi (media)": v[:, 1].mean(),
                  "potenza (effetti reali trovati)": v[:, 2].mean() / N_REALES,
                  "FDR (prop. di falsi tra le scoperte)": (v[:, 1] / np.maximum(v[:, 0], 1)).mean(),
                  "FWER (P di qualche falso)": (v[:, 1] > 0).mean()})
display(pd.DataFrame(filas).set_index("metodo").round(3))

**Cosa osservi?**

- Con gli 8 p-value: senza correzione se ne rifiutano 6; Bonferroni solo 1; BH, 4. Le ipotesi con p tra 0,012 e 0,049 sono quelle che separano i metodi.
- Senza correzione: ≈ 5,4 scoperte per studio (≈ 4,6 reali e ≈ 0,75 falsi); nel 54 % degli studi c'è almeno un falso positivo, e ≈ 12 % di ciò che si comunica è falso.
- Bonferroni: quasi nessun falso (FWER ≈ 4,6 %), ma trova solo ≈ 63 % degli effetti reali.
- BH: trova ≈ 81 % degli effetti reali e mantiene la FDR ≈ 3,8 % (sotto il 5 %), anche se in ≈ 18 % degli studi c'è qualche falso.
- **Ogni correzione costa potenza.** Bonferroni se **un solo** falso positivo è costoso; FDR se esplori molte ipotesi e verificherai poi i candidati.

*(La tua risposta qui.)*

---
## 8. Applicare le correzioni: Tukey, coppie di canali e molti sottogruppi

### 8.1 Post-hoc della lezione 8: Tukey corregge già

Tre magazzini con 40 ordini ciascuno (*esempio illustrativo*, costruiti come nella lezione 8). L'**HSD di Tukey** controlla già la FWER delle tre coppie: **non** va corretto una seconda volta. Se invece fai t-test di Welch a coppie, sì: la correzione la metti tu (Bonferroni o Holm).

In [ ]:
rng_datos = np.random.default_rng(42)
tiempo_a = rng_datos.normal(32, 6, 40)      # esempio illustrativo (minuti)
tiempo_b = rng_datos.normal(33, 6, 40)
tiempo_c = rng_datos.normal(38, 6, 40)
pares = [(0, 1), (0, 2), (1, 2)]
etiq = ["A − B", "A − C", "B − C"]
gs = [tiempo_a, tiempo_b, tiempo_c]

tk = stats.tukey_hsd(*gs)
p_welch = np.array([stats.ttest_ind(gs[i], gs[j], equal_var=False).pvalue for i, j in pares])
tabla_pares = pd.DataFrame({
    "Tukey (già corretto)": [tk.pvalue[i, j] for i, j in pares],
    "Welch senza correzione": p_welch,
    "Welch + Bonferroni": multipletests(p_welch, method="bonferroni")[1],
    "Welch + Holm": multipletests(p_welch, method="holm")[1]}, index=etiq)
display(tabla_pares.round(4))

**Cosa osservi?**

- A − B non si distingue con nessun metodo; A − C e B − C sì. Le conclusioni coincidono: Tukey e Welch corretto raccontano la stessa storia, con p corretti di grandezza simile.
- Ciò che **non** devi fare è applicare Bonferroni **sopra** i p-value già corretti di Tukey: sono già corretti e perderesti potenza senza motivo.
- Holm non dà mai un p corretto maggiore di Bonferroni.

*(La tua risposta qui.)*

### 8.2 Confrontare coppie di canali dopo un chi-quadro significativo

Il chi-quadro 3×2 della sezione 4 era significativo. Per sapere **quali canali** differiscono facciamo un chi-quadro 2×2 per ogni coppia (3 confronti) e correggiamo con Bonferroni e Holm. La famiglia sono le **3** coppie.

In [ ]:
from itertools import combinations
pares_c = list(combinations(range(3), 2))
p_pares = np.array([stats.chi2_contingency(obs3[[i, j]], correction=False).pvalue for i, j in pares_c])
res_pc = pd.DataFrame({
    "tassi (%)": [f"{tasas[i] * 100:.1f} vs {tasas[j] * 100:.1f}" for i, j in pares_c],
    "p senza correzione": p_pares,
    "p Bonferroni": multipletests(p_pares, method="bonferroni")[1],
    "p Holm": multipletests(p_pares, method="holm")[1]}, index=[f"{canales[i]} − {canales[j]}" for i, j in pares_c])
display(res_pc.round(4))

**Cosa osservi?**

- Social si distingue dal Motore di ricerca (p corretto ≈ 0,005) e da Email (≈ 0,03-0,05 a seconda del metodo); Email e Motore di ricerca non si distinguono (p ≈ 0,47).
- Con 3 confronti la correzione è lieve (Bonferroni moltiplica per 3); con 45 coppie la potenza ne risentirebbe molto di più.

*(La tua risposta qui.)*

### 8.3 Molti chi-quadro sui sottogruppi: il p-hacking in azione

Ora guardiamo la conversione di **email contro social in 10 segmenti di clienti** (150 visite per canale e segmento; dati sintetici). Costruiamo lo scenario: il tasso base è 20 % e **solo nei segmenti 1 e 2** i social convertono davvero di più (32 %); negli altri 8 non c'è differenza. Facciamo 10 chi-quadro 2×2 e vediamo cosa "esce" con e senza correzione. Il seme (148) è una replica concreta e illustrativa: in altre repliche i numeri cambiano.

In [ ]:
rng_seg = np.random.default_rng(148)
p_seg = []
for k in range(10):
    a = rng_seg.binomial(150, 0.20)                                  # canale 1: 20 % in tutti i segmenti
    b = rng_seg.binomial(150, 0.32 if k < 2 else 0.20)               # canale 2: 32 % solo nei segmenti 1 e 2
    p_seg.append(stats.chi2_contingency(np.array([[a, 150 - a], [b, 150 - b]]), correction=False).pvalue)
p_seg = np.array(p_seg)

rech_sc = p_seg < 0.05
rech_bf = multipletests(p_seg, method="bonferroni")[0]
rech_bh = multipletests(p_seg, method="fdr_bh")[0]
res_seg = pd.DataFrame({"p": p_seg, "effetto reale": np.arange(10) < 2,
                        "senza correzione": rech_sc, "Bonferroni": rech_bf, "BH": rech_bh},
                       index=[f"segmento {k + 1}" for k in range(10)])
display(res_seg.round(4))
real = np.arange(10) < 2
for nombre, r in [("senza correzione", rech_sc), ("Bonferroni", rech_bf), ("BH", rech_bh)]:
    print(f"{nombre:<18} scoperte = {r.sum()}   di cui falsi = {(r & ~real).sum()}")
print(f"Prob. teorica di qualche falso positivo con 8 nulli (senza correzione) = {1 - 0.95 ** 8:.3f}")

**Cosa osservi?**

- Senza correzione, 4 segmenti "escono" (p < 0,05): i due reali **e due falsi** (segmenti 3 e 8). Chi guardasse solo i p-value non corretti comunicherebbe "i social convertono di più nei segmenti 1, 2, 3 e 8", e metà sarebbe rumore.
- Con Bonferroni e con BH restano esattamente i due segmenti reali.
- Succede sempre: la probabilità di qualche falso con 8 nulli è 1 − 0,95⁸ = 0,337. **Decidi la famiglia (i 10 segmenti) prima di guardare** e correggi per tutto ciò che hai guardato, non per ciò che comunichi.

*(La tua risposta qui.)*

---
## 9. Quale test scelgo? Tabella decisionale (lezioni 7-9)

| Tipo di variabile risposta | Gruppi e disegno | Test abituale | Se i presupposti non valgono | Lezione |
|---|---|---|---|---|
| Numerica | 1 gruppo contro un valore di riferimento | t a un campione | Wilcoxon a un campione | 7 |
| Numerica | 2 gruppi indipendenti | t di Welch | Mann-Whitney | 7 |
| Numerica | 2 misure appaiate (prima/dopo) | t appaiato | Wilcoxon dei ranghi con segno | 7 |
| Numerica | ≥ 3 gruppi indipendenti | ANOVA + Tukey | Welch-ANOVA; Kruskal-Wallis | 8 |
| Categoriale (sì/no) | 2 gruppi indipendenti | z per due proporzioni (= χ² 2×2) | Fisher se attese < 5 | 7 e 9 |
| Categoriale | ≥ 3 gruppi, o variabile con ≥ 3 categorie | χ² di indipendenza + V di Cramér | Accorpare categorie (Fisher solo in 2×2) | 9 |
| Categoriale (una variabile) | Ripartizione osservata contro attesa | χ² di bontà di adattamento | Accorpare categorie | 9 |
| Categoriale (sì/no) | 2 misure appaiate (stessi soggetti) | McNemar (idea, non lo vediamo) | — | — |
| Qualsiasi | Molti test insieme | Correzione: Bonferroni/Holm (FWER) o BH (FDR) | Definire la famiglia prima | 9 |

> **Domande per decidere:** (1) che **tipo** di variabile misuro? (2) **quanti** gruppi? (3) **indipendenti o appaiati**? (4) valgono i **presupposti** (normalità, varianze, attese ≥ 5)? (5) quanti test faccio in **totale**?

---
## Riepilogo della lezione

- **Chi-quadro di indipendenza** (`chi2_contingency`) sui **conteggi**: attese = riga × colonna / totale, gdl = (r − 1)(c − 1); **V di Cramér** per la forza; residui per vedere dove; bontà di adattamento con `chisquare`.
- **z² = χ²** nel 2×2: il test per due proporzioni della lezione 7 e il chi-quadro senza correzione sono lo stesso test.
- **Attese ≥ 5**; altrimenti accorpa categorie o usa **Fisher** nel 2×2. Il chi-quadro senza correzione può rifiutare troppo con attese minuscole.
- **Confronti multipli:** P(qualche falso positivo) = 1 − (1 − α)^m (0,642 con m = 20). **Bonferroni/Holm** controllano la FWER; **BH** controlla la FDR ed è più potente. Ogni correzione costa potenza.
- **Applicazione:** Tukey è già corretto (non correggerlo di nuovo); con t-test o chi-quadro a coppie, correggi tu; con molti sottogruppi, definisci la famiglia **prima** di guardare.
- **Tabella decisionale:** tipo di variabile, numero di gruppi, indipendenti/appaiati, presupposti.

**Prossimo passo:** il notebook di pratica della lezione 9 e poi la lezione 10 (correlazione e regressione lineare). Il calcolo delle dimensioni campionarie e il test A/B arrivano nella lezione 13.